# Physics-Informed Neural Networks (PINNs) in PyTorch: Interactive Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jamshaidal/physics-informed-ml-framework/blob/main/notebooks/pinn_burgers_harmonic_tutorial.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-black?logo=github)](https://github.com/jamshaidal/physics-informed-ml-framework)

**Author:** Muhammad Jamshaid Ali  
**Focus:** Scientific Machine Learning (SciML) & Differential Equations  

---

## 1. Introduction & Mathematical Formulation

Traditional neural networks approximate relationships purely from training points. A **Physics-Informed Neural Network (PINN)** constrains the function space by embedding governing differential equations directly into the loss function via **automatic differentiation** (`torch.autograd.grad`):

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \lambda_{\text{pde}} \mathcal{L}_{\text{pde}} + \lambda_{\text{bc}} \mathcal{L}_{\text{bc}} + \lambda_{\text{ic}} \mathcal{L}_{\text{ic}}$$

In this tutorial, we:
1. Implement higher-order differential operators using PyTorch autograd.
2. Construct a **Fourier Feature PINN** to overcome spectral bias on high-frequency dynamics.
3. Solve the **Damped Harmonic Oscillator** boundary value problem and verify against exact analytical solutions.
4. Formulate the **1D Viscous Burgers' equation** for non-linear shock wave dynamics.

In [ ]:
# Step 1: Environment and PyTorch Setup
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set random seeds for exact reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using computational backend: {device}")

## 2. Automatic Differentiation for Higher-Order Derivatives
We construct a general utility function `compute_gradient` to calculate $\frac{\partial u}{\partial x}$ and $\frac{\partial^2 u}{\partial x^2}$ on computational graphs with `create_graph=True`.

In [ ]:
def compute_gradient(output_tensor, input_tensor):
    """
    Computes exact directional derivative d(output)/d(input) using PyTorch autograd.
    """
    grad = torch.autograd.grad(
        outputs=output_tensor,
        inputs=input_tensor,
        grad_outputs=torch.ones_like(output_tensor),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    return grad

# Verification test: f(x) = x^3 => f'(2) = 12, f''(2) = 12
x_test = torch.tensor([[2.0]], requires_grad=True)
f_test = x_test**3
df_dx = compute_gradient(f_test, x_test)
d2f_dx2 = compute_gradient(df_dx, x_test)

print(f"Test f(2)=2^3: df/dx = {df_dx.item():.1f}, d2f/dx2 = {d2f_dx2.item():.1f}")
assert np.isclose(df_dx.item(), 12.0) and np.isclose(d2f_dx2.item(), 12.0)
print("[PASS] Automatic differentiation higher-order gradient verified.")

## 3. Fourier Feature Neural Network Architecture
Standard multi-layer perceptrons suffer from spectral bias (favoring low frequencies). Random Fourier Features project coordinates into high-dimensional sinusoidal bases:
$$\gamma(x) = [\cos(2\pi B x), \sin(2\pi B x)]^T$$

In [ ]:
class FourierFeaturePINN(nn.Module):
    def __init__(self, in_dim=1, out_dim=1, hidden_dim=48, num_layers=4, fourier_dim=16, scale=5.0):
        super().__init__()
        self.B = nn.Parameter(torch.randn(in_dim, fourier_dim) * scale, requires_grad=False)
        layers = [nn.Linear(2 * fourier_dim, hidden_dim), nn.Tanh()]
        for _ in range(num_layers - 1):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.Tanh()])
        layers.append(nn.Linear(hidden_dim, out_dim))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        # Fourier coordinate mapping
        proj = 2.0 * np.pi * torch.matmul(x, self.B)
        feat = torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)
        return self.net(feat)

model = FourierFeaturePINN().to(device)
print(f"Constructed Fourier PINN with {sum(p.numel() for p in model.parameters() if p.requires_grad)} trainable parameters.")

## 4. Damped Harmonic Oscillator Problem
Consider the second-order ODE for an underdamped oscillator:
$$m \frac{d^2 x}{dt^2} + \mu \frac{dx}{dt} + k x = 0$$
with initial conditions $x(0) = 1.0, \; \dot{x}(0) = 0.0$ and physical parameters $m = 1.0, \; \mu = 0.4, \; k = 4.0$.

In [ ]:
# Physical constants
m, mu, k = 1.0, 0.4, 4.0
omega_0 = np.sqrt(k / m)
gamma = mu / (2.0 * m)
omega = np.sqrt(omega_0**2 - gamma**2)

def exact_solution(t):
    """Analytical ground truth"""
    return np.exp(-gamma * t) * (np.cos(omega * t) + (gamma / omega) * np.sin(omega * t))

# Collocation points across domain t in [0, 10]
t_colloc = torch.linspace(0.01, 10.0, 300, requires_grad=True).view(-1, 1).to(device)
t_ic = torch.tensor([[0.0]], requires_grad=True).to(device)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Beginning PINN training loop...")
loss_history = []

for epoch in range(1, 2001):
    optimizer.zero_grad()
    
    # 1. ODE residual at collocation points
    u = model(t_colloc)
    u_t = compute_gradient(u, t_colloc)
    u_tt = compute_gradient(u_t, t_colloc)
    ode_residual = m * u_tt + mu * u_t + k * u
    loss_ode = torch.mean(ode_residual**2)
    
    # 2. Initial conditions at t = 0
    u_0 = model(t_ic)
    u_t_0 = compute_gradient(u_0, t_ic)
    loss_ic = (u_0 - 1.0)**2 + (u_t_0 - 0.0)**2
    
    total_loss = loss_ode + 10.0 * loss_ic
    total_loss.backward()
    optimizer.step()
    
    loss_history.append(total_loss.item())
    if epoch % 400 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d} | Total Loss: {total_loss.item():.5e} | ODE: {loss_ode.item():.5e} | IC: {loss_ic.item():.5e}")

print("[PASS] Training converged successfully.")

## 5. Verification Against Analytical Ground Truth
Evaluating the relative $L_2$ error norm $\frac{\|u_{\text{pred}} - u_{\text{exact}}\|_2}{\|u_{\text{exact}}\|_2}$ and visualizing trajectory alignment.

In [ ]:
t_eval = torch.linspace(0, 10, 400).view(-1, 1).to(device)
with torch.no_grad():
    u_pred = model(t_eval).cpu().numpy().flatten()

t_eval_np = t_eval.cpu().numpy().flatten()
u_exact = exact_solution(t_eval_np)

# Relative L2 error
rel_l2 = np.linalg.norm(u_pred - u_exact) / np.linalg.norm(u_exact)
print(f"Relative L2 Error Norm: {rel_l2:.4e}")
assert rel_l2 < 0.05, f"Error exceeds tolerance: {rel_l2}"

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), dpi=140)

# Subplot 1: Trajectory Comparison
ax1.plot(t_eval_np, u_exact, 'k-', lw=2.2, label='Exact Analytical Solution')
ax1.plot(t_eval_np, u_pred, 'r--', lw=2.0, label='PINN Predicted Solution')
ax1.set_xlabel('Time $t$', fontsize=11)
ax1.set_ylabel('Displacement $x(t)$', fontsize=11)
ax1.set_title(f'Harmonic Oscillator ($L_2$ Rel. Error = {rel_l2:.2e})', fontsize=12)
ax1.legend(loc='upper right', frameon=False)
ax1.grid(True, linestyle=':', alpha=0.6)

# Subplot 2: Loss History
ax2.semilogy(loss_history, color='#0284c7', lw=1.8)
ax2.set_xlabel('Training Epochs', fontsize=11)
ax2.set_ylabel('Total Objective $\mathcal{L}$', fontsize=11)
ax2.set_title('Multi-Objective Loss Convergence', fontsize=12)
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()
print("[PASS] Benchmarking complete.")